Khai báo các thư viện

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime, timedelta

Cấu hình Spark và đọc dữ liệu từ iceberg

In [1]:
spark = SparkSession.builder \
    .master("spark://spark-master-service:7077") \
    .appName("dvc_fact_danhsachcongviechoso_donvi") \
    .config("spark.cores.max", "4") \
    .config("spark.executor.memory", "9g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

table = 'bdh_mdm.dvc_md_danhsachcongviechoso_otm'
df = spark.read \
    .format("iceberg") \
    .load(table)

24/10/23 15:56:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
df.printSchema()

root
 |-- macoquan: string (nullable = true)
 |-- sohoso: string (nullable = true)
 |-- ngaygiotiepnhan: timestamp_ntz (nullable = true)
 |-- macongviechoso: string (nullable = true)
 |-- maloaicongviec: string (nullable = true)
 |-- qdngay: double (nullable = true)
 |-- qdgio: double (nullable = true)
 |-- qdphut: double (nullable = true)
 |-- ngaygiobdcvhs: timestamp_ntz (nullable = true)
 |-- ngaygiobdcvhsonline: timestamp_ntz (nullable = true)
 |-- ngaygioktcvhs: timestamp_ntz (nullable = true)
 |-- ngaygioktcvhsonline: timestamp_ntz (nullable = true)
 |-- ngaygioktcvhsdh: timestamp_ntz (nullable = true)
 |-- trangthaicongviechoso: integer (nullable = true)
 |-- madonvithuchien: string (nullable = true)
 |-- matochuyenmonthuchien: string (nullable = true)
 |-- macanbothuchien: string (nullable = true)
 |-- mathutuc: string (nullable = true)
 |-- tenthutuc: string (nullable = true)
 |-- malinhvuc: string (nullable = true)
 |-- tenlinhvuc: string (nullable = true)
 |-- tenloaicongvie

Lấy dữ liệu ngày hôm qua

In [13]:
# n_days = -1
# day = (datetime.now().date() + timedelta(days=n_days)).strftime("%Y-%m-%d")
# df = df.filter(df['ngaygioktcvhs'].contains(day))

Chuyển đổi một số cột về dạng timestamp

In [23]:
df = df.withColumn('ngaygiotiepnhan', to_timestamp('ngaygiotiepnhan', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygiotiepnhan",col("ngaygiotiepnhan") - expr("INTERVAL 7 HOURS"))

df = df.withColumn('ngaygiobdcvhs', to_timestamp('ngaygiobdcvhs', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygiobdcvhs",col("ngaygiobdcvhs") - expr("INTERVAL 7 HOURS"))

df = df.withColumn('ngaygiobdcvhsonline', to_timestamp('ngaygiobdcvhsonline', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygiobdcvhsonline",col("ngaygiobdcvhsonline") - expr("INTERVAL 7 HOURS"))

df = df.withColumn('ngaygioktcvhs', to_timestamp('ngaygioktcvhs', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygioktcvhs",col("ngaygioktcvhs") - expr("INTERVAL 7 HOURS"))

df = df.withColumn('ngaygioktcvhsonline', to_timestamp('ngaygioktcvhsonline', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygioktcvhsonline",col("ngaygioktcvhsonline") - expr("INTERVAL 7 HOURS"))

df = df.withColumn('ngaygioktcvhsdh', to_timestamp('ngaygioktcvhsdh', 'yyyy-MM-dd HH:mm:ss'))
df = df.withColumn("ngaygioktcvhsdh",col("ngaygioktcvhsdh") - expr("INTERVAL 7 HOURS"))

Xử lý tạo một số cột mới

In [24]:
df = df.withColumn('ngaybdcvhs', split(df['ngaygiobdcvhs'], ' ').getItem(0))

In [25]:
df = df.withColumn('dunghan',
                   when((col('ngaygioktcvhsdh').isNotNull()) & (col('ngaygioktcvhs').isNotNull()),
                   when(col('ngaygioktcvhsdh') >= col('ngaygioktcvhs'), 1).otherwise(0))
                   .otherwise(None))

df = df.withColumn('onlinedunghan',
                   when((col('ngaygioktcvhsdh').isNotNull()) & (col('ngaygioktcvhsonline').isNotNull()),
                   when(col('ngaygioktcvhsdh') >= col('ngaygioktcvhsonline'), 1).otherwise(0))
                   .otherwise(None))

In [26]:
df = df.withColumn('tgluugiucvhs', round((col('ngaygioktcvhs').cast("long") - col('ngaygiotiepnhan').cast("long")) / 86400, 5))
df = df.withColumn('tgchobatdaucvhs', round((col('ngaygiobdcvhs').cast("long") - col('ngaygiotiepnhan').cast("long")) / 86400, 5))
df = df.withColumn('tgxulycvhs', round((col('ngaygioktcvhs').cast("long") - col('ngaygiobdcvhs').cast("long")) / 86400, 5))
df = df.withColumn('tgxulycvhsonline', round((col('ngaygioktcvhsonline').cast("long") - col('ngaygiobdcvhsonline').cast("long")) / 86400, 5))

In [27]:
result = df.groupBy('maloaicongviec', 'trangthaicongviechoso', 'matochuyenmonthuchien', 'macanbothuchien',\
                    'malinhvuc', 'ngaybdcvhs') \
           .agg(
               count('*').alias('tongsocvhs'),
               sum(when(col('dunghan') == 1, 1).otherwise(0)).alias('slcvhsdunghan'),
               sum(when(col('dunghan') == 0, 1).otherwise(0)).alias('slcvhsquahan'),
               sum(when(col('onlinedunghan') == 1, 1).otherwise(0)).alias('slcvhsonlinedunghan'),
               sum(when(col('onlinedunghan') == 0, 1).otherwise(0)).alias('slcvhsonlinequahan'),
               sum(when(col('dunghan').isNull(), 1).otherwise(0)).alias('cvhskhac'),
               sum(when(col('dunghan') == 1, col('tgxulycvhs')).otherwise(None)).alias('tgtbxulycvhsdunghan'),
               sum(when(col('dunghan') == 0, col('tgxulycvhs')).otherwise(None)).alias('tgtbxulycvhsquahan'),
               sum(when(col('onlinedunghan') == 1, col('tgxulycvhsonline')).otherwise(None)).alias('tgtbxulycvhsonlinedunghan'),
               sum(when(col('onlinedunghan') == 0, col('tgxulycvhsonline')).otherwise(None)).alias('tgtbxulycvhsonlinequahan'),
               sum('tgluugiucvhs').alias('tgluugiucvhs'),
               sum('tgchobatdaucvhs').alias('tgchobatdaucvhs'),
               sum('tgxulycvhs').alias('tgxulycvhs'),
               sum('tgxulycvhsonline').alias('tgxulycvhsonline')
           ) 

result = result.withColumn('ngaybdcvhs', to_date('ngaybdcvhs', 'yyyy-MM-dd'))
result = result.withColumn('tgtbxulycvhsdunghan', round(col('tgtbxulycvhsdunghan')/col('slcvhsdunghan'), 2))
result = result.withColumn('tgtbxulycvhsquahan', round(abs(col('tgtbxulycvhsquahan'))/col('slcvhsquahan'), 2))
result = result.withColumn('tgtbxulycvhsonlinedunghan', round(col('tgtbxulycvhsonlinedunghan')/col('slcvhsonlinedunghan'), 2))
result = result.withColumn('tgtbxulycvhsonlinequahan', round(abs(col('tgtbxulycvhsonlinequahan'))/col('slcvhsonlinequahan'), 2))
result = result.withColumn('tgluugiucvhs', round(col('tgluugiucvhs'), 2))
result = result.withColumn('tgchobatdaucvhs', round(col('tgchobatdaucvhs'), 2))
result = result.withColumn('tgxulycvhs', round(col('tgxulycvhs'), 2))
result = result.withColumn('tgxulycvhsonline', round(col('tgxulycvhsonline'), 2))

In [ ]:
result = result.withColumn('ngaycapnhat', current_timestamp())
result = result.withColumn('maky', date_format(current_date(), 'yyyyMM'))

result.write \
    .format("jdbc") \
    .option("driver","com.mysql.cj.jdbc.Driver") \
    .option("url", "jdbc:mysql://10.168.6.106:3306/bdh_dtm") \
    .option("dbtable", "dvc_fact_danhsachcongviechoso_donvi") \
    .option("user", "acc_etl") \
    .option("password", "Vnpt123456") \
    .mode("append") \
    .save()

In [6]:
spark.stop()